# GSE268904: extract, inventory and QC (R, Google Colab)

Human subcutaneous adipose snRNA-seq (10x), 14 samples: lean (4), obese (7), obese diabetic (3).
This notebook extracts `GSE268904_RAW.tar`, builds a checksummed file inventory, reads each sample's 10x matrix, reports nuclei counts and QC metrics, and makes a **crude** endothelial estimate. An optional last section clusters and annotates for a better endothelial count.

**Before running**
1. Runtime > Change runtime type > **R**.
2. Open the folder icon on the left, click the Drive icon to **mount Google Drive**.
3. Run the cells in order. Only the small output files (CSV, markdown) need to be shared; the raw data stay in Colab.

*Status: written without being able to run R, so expect small fixes on a first run. Paste any error message back to the assistant.*

## 1. Configuration

In [ ]:
# Edit TAR_PATH after you find the file in the next cell
TAR_PATH <- ""                                   # e.g. "/content/drive/MyDrive/GSE268904_RAW.tar"
WORK_DIR <- "/content/gse268904_raw"             # extracted files (treated as read-only)
OUT_DIR  <- "/content/gse268904_qc"              # small outputs to download or copy to Drive
MIN_UMI  <- 500                                  # barcodes below this total UMI are not counted as nuclei
ENDO_MARKERS <- c("PECAM1", "VWF", "CLDN5", "EGFL7", "FLT1", "PTPRB")  # crude endothelial panel
GENES_EXTRA <- character(0)                      # extra genes whose raw counts you also want per nucleus, e.g. c("ADIPOQ", "PDGFRA")
ENDO_MIN_MARKERS <- 3                            # specific call: at least this many of the 6 core endothelial genes detected
ENDO_OVER_OTHER  <- 2                            # ... and endothelial score >= this many times the best non-endothelial panel score
PANELS <- list(                                  # lineage marker panels (scored per nucleus as counts per 10,000 UMI)
  endothelial           = c(ENDO_MARKERS, "CDH5", "KDR"),
  lymphatic_endothelial = c("PROX1", "LYVE1", "FLT4"),
  adipocyte             = c("ADIPOQ", "PLIN1", "LIPE", "LEP"),
  fibroblast_progenitor = c("PDGFRA", "DCN", "LUM", "COL1A1"),
  immune                = c("PTPRC", "CD163", "MRC1", "CD14"),
  mural                 = c("ACTA2", "RGS5", "MYH11", "PDGFRB"))
RUN_SEURAT <- FALSE                              # TRUE = also run the optional clustering section (slow)
dir.create(WORK_DIR, recursive = TRUE, showWarnings = FALSE)
dir.create(OUT_DIR,  recursive = TRUE, showWarnings = FALSE)

## 2. Find the tar in Drive

In [ ]:
hits <- list.files("/content/drive", pattern = "GSE268904", recursive = TRUE, full.names = TRUE)
print(hits)
# Pick the .tar path and set TAR_PATH in the previous cell, or do it here:
if (TAR_PATH == "") { tars <- hits[grepl("\\.tar$", hits)]; if (length(tars) >= 1) TAR_PATH <- tars[1] }
cat("Using:", TAR_PATH, "\n")


### 2b. If Drive cannot be mounted in the R runtime
Download the file directly instead (works only if the link is shared as "anyone with the link"). Skip this cell if section 2 found the file.


In [ ]:
if (TAR_PATH == "" || !file.exists(TAR_PATH)) {
  system("pip install -q gdown")
  system("gdown 1mKaM2c4473gV3zCoCV3F1ZknlCIDlAwq -O /content/GSE268904_RAW.tar")
  TAR_PATH <- "/content/GSE268904_RAW.tar"
}
stopifnot(file.exists(TAR_PATH)); cat("Using:", TAR_PATH, "\n")


## 3. Install packages

In [ ]:
need <- c("Matrix", "data.table", "digest")
for (p in need) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, repos = "https://cloud.r-project.org")
suppressPackageStartupMessages({library(Matrix); library(data.table); library(digest)})

## 4. Extract the archive and build the file inventory (SHA-256)

In [ ]:
untar(TAR_PATH, exdir = WORK_DIR)
files <- list.files(WORK_DIR, recursive = TRUE, full.names = TRUE)
files <- files[!file.info(files)$isdir]
inv <- data.table(
  relative_path = sub(paste0("^", WORK_DIR, "/?"), "", files),
  filename = basename(files),
  size_bytes = file.info(files)$size,
  sha256 = vapply(files, function(f) digest(f, algo = "sha256", file = TRUE), character(1)),
  file_type = ifelse(grepl("barcodes", files), "barcodes",
              ifelse(grepl("features|genes", files), "features",
              ifelse(grepl("matrix", files), "matrix", "other")))
)
inv[, gsm := sub("^(GSM[0-9]+).*", "\\1", filename)]
inv[, duplicate_of_n := .N, by = sha256]
cat("Files extracted:", nrow(inv), " (expected 42 = 14 samples x 3 files)\n")
print(inv[, .N, by = file_type])
print(inv[duplicate_of_n > 1, .(files_with_identical_content = .N), by = file_type])
cat("Identical features.tsv.gz files across samples are expected (same gene list); identical barcodes or matrix files would be suspicious.\n")


## 5. Sample metadata (copied from the GEO sample records)

In [ ]:
meta <- data.table(
  gsm = sprintf("GSM%d", 8302405:8302418),
  sample = c("9S","10S","13S","17S","SAT22","SAT54","SAT1","SAT18","SAT34","SAT39","SAT4","SAT51","SAT58","SAT60"),
  condition = c("obese","obese diabetic","lean","obese diabetic","obese diabetic","lean","obese","obese","obese","obese","obese","obese","lean","lean"),
  sex = c("female","female","female","female","male","female","male","female","male","male","male","female","male","male"),
  age = c(37,41,42,34,50,61,43,29,40,41,30,44,54,26),
  bmi = c(39.5,61.5,23,52.6,39.7,22.3,41.7,61.3,62.5,61.2,49.7,57,24.3,23.5))
# Referential check: every metadata GSM must have 3 files
chk <- inv[, .(n_files = .N, has_barcodes = any(file_type == "barcodes"), has_features = any(file_type == "features"), has_matrix = any(file_type == "matrix")), by = gsm]
print(merge(meta[, .(gsm, sample)], chk, by = "gsm", all = TRUE))

## 6. Per-sample nuclei counts, QC metrics and crude endothelial estimate

In [ ]:
mito_prefix <- "^MT-"
res <- list(); bt_list <- list()
for (i in seq_len(nrow(meta))) {
  gsm <- meta$gsm[i]; smp <- meta$sample[i]
  pick <- function(type) files[grepl(gsm, files) & grepl(type, files)]
  fb <- pick("barcodes"); ff <- pick("features|genes"); fm <- pick("matrix")
  if (length(fb) != 1 || length(ff) != 1 || length(fm) != 1) { res[[i]] <- data.table(gsm = gsm, sample = smp, status = "missing files"); next }
  m <- tryCatch(readMM(fm), error = function(e) NULL)
  if (is.null(m)) { res[[i]] <- data.table(gsm = gsm, sample = smp, status = "matrix unreadable"); next }
  bc <- fread(fb, header = FALSE)[[1]]; feat <- fread(ff, header = FALSE); gene <- feat[[2]]
  if (ncol(m) != length(bc) || nrow(m) != length(gene)) { res[[i]] <- data.table(gsm = gsm, sample = smp, status = "dimension mismatch"); next }
  m <- as(m, "CsparseMatrix")
  umi <- Matrix::colSums(m); ngene <- Matrix::colSums(m > 0)
  mt <- Matrix::colSums(m[grepl(mito_prefix, gene), , drop = FALSE]) / pmax(umi, 1)
  present <- ENDO_MARKERS[ENDO_MARKERS %in% gene]
  det <- if (length(present)) Matrix::colSums(m[match(present, gene), , drop = FALSE] > 0) else rep(0, ncol(m))
  keep <- umi >= MIN_UMI
  # one row per barcode, none dropped
  gp <- unique(c(unlist(PANELS), GENES_EXTRA)); gp <- gp[gp %in% gene]
  sc <- sapply(PANELS, function(g) { g <- g[g %in% gene]; if (length(g)) Matrix::colSums(m[match(g, gene), , drop = FALSE]) / pmax(umi, 1) * 1e4 else rep(0, ncol(m)) })
  if (is.null(dim(sc))) sc <- matrix(sc, ncol = length(PANELS))
  colnames(sc) <- names(PANELS)
  top <- ifelse(rowSums(sc) == 0, "none", colnames(sc)[max.col(sc, ties.method = "first")])
  best_other <- apply(sc[, colnames(sc) != "endothelial", drop = FALSE], 1, max)
  endo_spec <- keep & det >= ENDO_MIN_MARKERS & top == "endothelial" & sc[, "endothelial"] > 0 & sc[, "endothelial"] >= ENDO_OVER_OTHER * best_other
  sdt <- as.data.table(round(sc, 3)); setnames(sdt, paste0("score_", colnames(sc)))
  bt <- data.table(row_type = "barcode", gsm = gsm, sample = smp, condition = meta$condition[i], sex = meta$sex[i], age = meta$age[i], bmi = meta$bmi[i],
    barcode = bc, total_umi = as.numeric(umi), n_genes_detected = as.integer(ngene), mito_frac = round(as.numeric(mt), 5),
    n_endo_markers_detected = as.integer(det), passes_min_umi = keep,
    endo_ge2 = keep & det >= 2, endo_ge3 = keep & det >= 3, endo_ge4 = keep & det >= 4, top_lineage = top, endo_specific = endo_spec)
  bt <- cbind(bt, sdt)
  if (length(gp)) { cnt <- as.matrix(Matrix::t(m[match(gp, gene), , drop = FALSE])); colnames(cnt) <- paste0("count_", gp); bt <- cbind(bt, as.data.table(cnt)) }
  bt_list[[i]] <- bt
  res[[i]] <- data.table(gsm = gsm, sample = smp, condition = meta$condition[i], status = "ok",
    n_barcodes_file = length(bc), n_genes_file = length(gene), n_nuclei_umi_ge_min = sum(keep),
    median_umi = median(umi[keep]), median_genes = median(ngene[keep]), median_mito_frac = round(median(mt[keep]), 4),
    raw_counts_integer = all(m@x == round(m@x)),
    markers_present = paste(present, collapse = "|"),
    endo_ge2 = sum(keep & det >= 2), endo_ge3 = sum(keep & det >= 3), endo_ge4 = sum(keep & det >= 4), endo_specific = sum(endo_spec))
  cat(smp, ":", length(bc), "barcodes;", sum(keep), "with UMI >=", MIN_UMI, "\n")
  rm(m); gc(verbose = FALSE)
}
per_sample <- rbindlist(res, fill = TRUE)
print(per_sample)

## 7. Group totals

In [ ]:
ok <- per_sample[status == "ok"]
group <- ok[, .(n_samples = .N, nuclei = sum(n_nuclei_umi_ge_min), endo_ge2 = sum(endo_ge2), endo_ge3 = sum(endo_ge3), endo_ge4 = sum(endo_ge4)), by = condition]
overall <- data.table(condition = "ALL", n_samples = nrow(ok), nuclei = sum(ok$n_nuclei_umi_ge_min), endo_ge2 = sum(ok$endo_ge2), endo_ge3 = sum(ok$endo_ge3), endo_ge4 = sum(ok$endo_ge4))
group <- rbind(group, overall)
print(group)
cat("\nCRUDE estimate: nuclei where >= k of the endothelial markers are detected. Not a cell-type annotation.\n")

## 8. (Optional) Seurat clustering for a better endothelial count
Set `RUN_SEURAT <- TRUE` in section 1 and re-run from here. This installs Seurat (can take 10+ minutes) and clusters each sample separately. Endothelial clusters are called when the cluster's median module score is high and PECAM1/VWF/CLDN5 are detected in a large fraction of its nuclei. **Check the output by eye** before trusting the counts. Per-sample clustering is a shortcut; a proper analysis integrates samples with batch in the model.

In [ ]:
if (RUN_SEURAT) {
  if (!requireNamespace("Seurat", quietly = TRUE)) install.packages("Seurat", repos = "https://cloud.r-project.org")
  suppressPackageStartupMessages(library(Seurat))
  seu_res <- list(); bt_list <- list()
  for (i in seq_len(nrow(meta))) {
    gsm <- meta$gsm[i]; smp <- meta$sample[i]
    pick <- function(type) files[grepl(gsm, files) & grepl(type, files)]
    m <- readMM(pick("matrix")); bc <- fread(pick("barcodes"), header = FALSE)[[1]]; gene <- fread(pick("features|genes"), header = FALSE)[[2]]
    rownames(m) <- make.unique(gene); colnames(m) <- bc
    obj <- CreateSeuratObject(m, min.cells = 3, min.features = 200)
    obj[["percent.mt"]] <- PercentageFeatureSet(obj, pattern = "^MT-")
    obj <- subset(obj, subset = nCount_RNA >= MIN_UMI & percent.mt < 10)
    obj <- NormalizeData(obj, verbose = FALSE); obj <- FindVariableFeatures(obj, nfeatures = 2000, verbose = FALSE)
    obj <- ScaleData(obj, verbose = FALSE); obj <- RunPCA(obj, npcs = 30, verbose = FALSE)
    obj <- FindNeighbors(obj, dims = 1:20, verbose = FALSE); obj <- FindClusters(obj, resolution = 0.5, verbose = FALSE)
    genes_ok <- intersect(ENDO_MARKERS, rownames(obj))
    obj <- AddModuleScore(obj, features = list(genes_ok), name = "endo", verbose = FALSE)
    key <- intersect(c("PECAM1", "VWF", "CLDN5"), rownames(obj))
    pct <- sapply(levels(obj$seurat_clusters), function(cl) mean(colSums(GetAssayData(obj, layer = "counts")[key, obj$seurat_clusters == cl, drop = FALSE] > 0) >= 1))
    sc  <- tapply(obj$endo1, obj$seurat_clusters, median)
    endo_clusters <- names(sc)[sc > 0.3 & pct[names(sc)] > 0.3]
    n_endo <- sum(obj$seurat_clusters %in% endo_clusters)
    seu_res[[i]] <- data.table(sample = smp, condition = meta$condition[i], nuclei_after_qc = ncol(obj), n_clusters = length(levels(obj$seurat_clusters)),
                               endo_clusters = paste(endo_clusters, collapse = "|"), endothelial_nuclei = n_endo)
    cat(smp, ": ", ncol(obj), "nuclei; endothelial clusters:", paste(endo_clusters, collapse = ","), "->", n_endo, "\n")
    rm(obj, m); gc(verbose = FALSE)
  }
  seu <- rbindlist(seu_res); print(seu)
  print(seu[, .(nuclei = sum(nuclei_after_qc), endothelial = sum(endothelial_nuclei)), by = condition])
} else cat("Skipped (RUN_SEURAT = FALSE)\n")

## 9. Consolidate everything into ONE csv and save
One row for **every barcode** in every sample (none dropped; `passes_min_umi` flags low-UMI droplets), with sample metadata, QC metrics and marker counts, followed by sample-summary, group and overall rows. `row_type` tells them apart.


In [ ]:
# ---- wide file info per sample (from the inventory) ----
fi <- dcast(inv, gsm ~ file_type, value.var = c("filename", "size_bytes", "sha256"))
# ---- per-sample table ----
final <- merge(meta, fi, by = "gsm", all.x = TRUE)
final <- merge(final, per_sample[, !c("sample", "condition"), with = FALSE], by = "gsm", all.x = TRUE)
if (exists("seu")) {
  s2 <- copy(seu)[, .(sample, seurat_nuclei_after_qc = nuclei_after_qc, seurat_n_clusters = n_clusters, seurat_endothelial_clusters = endo_clusters, seurat_endothelial_nuclei = endothelial_nuclei)]
  final <- merge(final, s2, by = "sample", all.x = TRUE)
}
final[, row_type := "sample_summary"]
# ---- group and overall rows ----
num <- c("n_barcodes_file", "n_nuclei_umi_ge_min", "endo_ge2", "endo_ge3", "endo_ge4", "endo_specific", "seurat_nuclei_after_qc", "seurat_endothelial_nuclei")
num <- intersect(num, names(final))
grp <- final[, c(list(n_samples = .N), lapply(.SD, sum, na.rm = TRUE)), by = condition, .SDcols = num][, row_type := "group"]
ovr <- final[, c(list(n_samples = .N), lapply(.SD, sum, na.rm = TRUE)), .SDcols = num][, `:=`(row_type = "overall", condition = "ALL")]
ovr[, notes := paste("Nuclei = barcodes with total UMI >=", MIN_UMI, "; endo_geK = nuclei with >= K of 6 endothelial genes detected (CRUDE estimate, not an annotation); endo_specific = >= ENDO_MIN_MARKERS core genes AND endothelial panel is the top lineage AND >= ENDO_OVER_OTHER x the best other panel (still marker-based, not a cluster annotation); ambient RNA can inflate endo_ge2;",
                     "identical features files across samples are expected; no data were removed; processed", format(Sys.time(), "%Y-%m-%d"))]
summary_tbl <- rbindlist(list(final, grp, ovr), fill = TRUE)
barcodes_tbl <- rbindlist(bt_list, fill = TRUE)
final <- rbindlist(list(barcodes_tbl, summary_tbl), fill = TRUE)
lead <- c("row_type", "gsm", "sample", "condition", "sex", "age", "bmi", "barcode", "n_samples", "status")
setcolorder(final, c(intersect(lead, names(final)), setdiff(names(final), lead)))
out_csv <- file.path(OUT_DIR, "GSE268904_consolidated.csv")
fwrite(final, out_csv)
cat("Wrote", out_csv, "|", nrow(final), "rows x", ncol(final), "columns |", round(file.info(out_csv)$size / 1e3, 1), "KB\n")
cat("Barcode rows written:", nrow(barcodes_tbl), "| barcodes in source files:", sum(per_sample$n_barcodes_file, na.rm = TRUE), "\n")
stopifnot(nrow(barcodes_tbl) == sum(per_sample$n_barcodes_file, na.rm = TRUE))
print(final[row_type %in% c("group", "overall"), c("row_type", "condition", "n_samples", "n_nuclei_umi_ge_min", "endo_ge3", "endo_specific"), with = FALSE])
# Copy to Drive
dest <- "/content/drive/MyDrive/gse268904_qc"
dir.create(dest, showWarnings = FALSE); file.copy(out_csv, dest, overwrite = TRUE)
cat("Copied to", dest, "- upload GSE268904_consolidated.csv back to the chat\n")